In [1]:
# Import required libraries
from itertools import combinations
from collections import defaultdict
import numpy as np
from scipy.spatial import distance

# Custom Apriori Algorithm
def apriori(transactions, min_support):
    # Count individual items
    item_count = defaultdict(int)
    for transaction in transactions:
        for item in transaction:
            item_count[item] += 1
    
    # Find frequent 1-itemsets
    n_transactions = len(transactions)
    min_support_count = min_support * n_transactions
    frequent_items = {frozenset([item]): count for item, count in item_count.items() if count >= min_support_count}
    
    # Generate frequent itemsets of increasing size
    k = 2
    while frequent_items:
        # Generate candidate k-itemsets
        candidates = defaultdict(int)
        for transaction in transactions:
            transaction_set = set(transaction)
            for itemset in combinations(transaction_set, k):
                candidates[frozenset(itemset)] += 1
        
        # Find frequent k-itemsets
        frequent_k = {itemset: count for itemset, count in candidates.items() if count >= min_support_count}
        if not frequent_k:
            break
        
        frequent_items.update(frequent_k)
        k += 1
    
    return frequent_items

# Custom FP-Growth Algorithm
class FPNode:
    def __init__(self, item, count, parent):
        self.item = item
        self.count = count
        self.parent = parent
        self.children = {}
        self.node_link = None

class FPTree:
    def __init__(self):
        self.root = FPNode(None, 0, None)
        self.header_table = defaultdict(list)
        
def fp_growth(transactions, min_support):
    # Count item frequencies
    item_count = defaultdict(int)
    for transaction in transactions:
        for item in transaction:
            item_count[item] += 1
    
    # Filter items by minimum support
    n_transactions = len(transactions)
    min_support_count = min_support * n_transactions
    frequent_items = {item: count for item, count in item_count.items() if count >= min_support_count}
    
    # Build FP-Tree
    fp_tree = FPTree()
    for transaction in transactions:
        # Filter transaction to include only frequent items
        filtered_transaction = [item for item in transaction if item in frequent_items]
        if not filtered_transaction:
            continue
        # Sort by frequency (descending)
        filtered_transaction.sort(key=lambda x: frequent_items[x], reverse=True)
        
        # Insert into tree
        current_node = fp_tree.root
        for item in filtered_transaction:
            if item in current_node.children:
                current_node.children[item].count += 1
            else:
                new_node = FPNode(item, 1, current_node)
                current_node.children[item] = new_node
                fp_tree.header_table[item].append(new_node)
            current_node = current_node.children[item]
    
    # Mine frequent patterns
    frequent_itemsets = {}
    def mine_patterns(header_table, min_support_count, prefix):
        for item in header_table:
            new_prefix = prefix | {item}
            frequent_itemsets[frozenset(new_prefix)] = sum(node.count for node in header_table[item])
            
            # Create conditional pattern base
            conditional_base = []
            for node in header_table[item]:
                count = node.count
                path = []
                current = node.parent
                while current.item is not None:
                    path.append(current.item)
                    current = current.parent
                if path:
                    conditional_base.extend([path] * count)
            
            # Build conditional FP-tree and mine recursively
            if conditional_base:
                conditional_tree = FPTree()
                item_counts = defaultdict(int)
                for path in conditional_base:
                    for item in path:
                        item_counts[item] += 1
                
                # Filter by support
                filtered_items = {item for item, count in item_counts.items() if count >= min_support_count}
                for path in conditional_base:
                    filtered_path = [item for item in path if item in filtered_items]
                    if not filtered_path:
                        continue
                    filtered_path.sort(key=lambda x: item_counts[x], reverse=True)
                    current_node = conditional_tree.root
                    for item in filtered_path:
                        if item in current_node.children:
                            current_node.children[item].count += 1
                        else:
                            new_node = FPNode(item, 1, current_node)
                            current_node.children[item] = new_node
                            conditional_tree.header_table[item].append(new_node)
                        current_node = current_node.children[item]
                
                mine_patterns(conditional_tree.header_table, min_support_count, new_prefix)
    
    mine_patterns(fp_tree.header_table, min_support_count, set())
    return frequent_itemsets

# Custom Distance Calculations
def custom_euclidean(point1, point2):
    return np.sqrt(sum((a - b) ** 2 for a, b in zip(point1, point2)))

def custom_manhattan(point1, point2):
    return sum(abs(a - b) for a, b in zip(point1, point2))

def custom_chebyshev(point1, point2):
    return max(abs(a - b) for a, b in zip(point1, point2))

# Example Usage
if __name__ == "__main__":
    # Sample transaction data
    transactions = [
        ['milk', 'bread', 'butter'],
        ['milk', 'bread'],
        ['milk', 'butter'],
        ['bread', 'butter'],
        ['milk', 'bread', 'butter', 'cheese']
    ]
    min_support = 0.4  # 40% support threshold

    # Run Apriori
    apriori_result = apriori(transactions, min_support)
    print("Apriori Frequent Itemsets:")
    for itemset, count in apriori_result.items():
        print(f"{set(itemset)}: {count}")

    # Run FP-Growth
    fp_growth_result = fp_growth(transactions, min_support)
    print("\nFP-Growth Frequent Itemsets:")
    for itemset, count in fp_growth_result.items():
        print(f"{set(itemset)}: {count}")

    # Distance calculations
    point1 = [2, 3]
    point2 = [5, 7]

    # Custom distance calculations
    print("\nCustom Distance Calculations:")
    print(f"Euclidean Distance: {custom_euclidean(point1, point2):.2f}")
    print(f"Manhattan Distance: {custom_manhattan(point1, point2):.2f}")
    print(f"Chebyshev Distance: {custom_chebyshev(point1, point2):.2f}")

    # Scipy distance calculations
    print("\nScipy Distance Calculations:")
    print(f"Euclidean Distance: {distance.euclidean(point1, point2):.2f}")
    print(f"Manhattan Distance: {distance.cityblock(point1, point2):.2f}")
    print(f"Chebyshev Distance: {distance.chebyshev(point1, point2):.2f}")

Apriori Frequent Itemsets:
{'milk'}: 4
{'bread'}: 4
{'butter'}: 4
{'bread', 'milk'}: 3
{'bread', 'butter'}: 3
{'milk', 'butter'}: 3
{'bread', 'milk', 'butter'}: 2

FP-Growth Frequent Itemsets:
{'milk'}: 4
{'bread'}: 4
{'bread', 'milk'}: 3
{'butter'}: 4
{'bread', 'butter'}: 3
{'milk', 'butter'}: 3
{'bread', 'milk', 'butter'}: 2

Custom Distance Calculations:
Euclidean Distance: 5.00
Manhattan Distance: 7.00
Chebyshev Distance: 4.00

Scipy Distance Calculations:
Euclidean Distance: 5.00
Manhattan Distance: 7.00
Chebyshev Distance: 4.00


In [4]:

from itertools import combinations
from collections import defaultdict
import numpy as np
from scipy.spatial import distance

# Custom Apriori Algorithm
def apriori(transactions, min_support):
    n_transactions = len(transactions)
    min_support_count = min_support * n_transactions
    
    # Count individual items
    item_count = defaultdict(int)
    for transaction in transactions:
        for item in transaction:
            item_count[item] += 1
    
    # Frequent 1-itemsets
    frequent_items = {frozenset([item]): count for item, count in item_count.items() if count >= min_support_count}
    
    # Generate frequent k-itemsets
    k = 2
    while frequent_items:
        candidates = defaultdict(int)
        for transaction in transactions:
            for itemset in combinations(set(transaction), k):
                candidates[frozenset(itemset)] += 1
        frequent_k = {itemset: count for itemset, count in candidates.items() if count >= min_support_count}
        if not frequent_k:
            break
        frequent_items.update(frequent_k)
        k += 1
    return frequent_items

# Custom FP-Growth Algorithm
class FPNode:
    def __init__(self, item, count, parent):
        self.item = item
        self.count = count
        self.parent = parent
        self.children = {}
        self.node_link = None

class FPTree:
    def __init__(self):
        self.root = FPNode(None, 0, None)
        self.header_table = defaultdict(list)

def fp_growth(transactions, min_support):
    n_transactions = len(transactions)
    min_support_count = min_support * n_transactions
    
    # Item frequencies
    item_count = defaultdict(int)
    for transaction in transactions:
        for item in transaction:
            item_count[item] += 1
    frequent_items = {item: count for item, count in item_count.items() if count >= min_support_count}
    
    # Build FP-Tree
    fp_tree = FPTree()
    for transaction in transactions:
        filtered_transaction = [item for item in transaction if item in frequent_items]
        if filtered_transaction:
            filtered_transaction.sort(key=lambda x: frequent_items[x], reverse=True)
            current_node = fp_tree.root
            for item in filtered_transaction:
                if item in current_node.children:
                    current_node.children[item].count += 1
                else:
                    new_node = FPNode(item, 1, current_node)
                    current_node.children[item] = new_node
                    fp_tree.header_table[item].append(new_node)
                current_node = current_node.children[item]
    
    # Mine frequent patterns
    frequent_itemsets = {}
    def mine_patterns(header_table, min_support_count, prefix):
        for item in header_table:
            new_prefix = prefix | {item}
            frequent_itemsets[frozenset(new_prefix)] = sum(node.count for node in header_table[item])
            conditional_base = []
            for node in header_table[item]:
                count = node.count
                path = []
                current = node.parent
                while current.item is not None:
                    path.append(current.item)
                    current = current.parent
                if path:
                    conditional_base.extend([path] * count)
            if conditional_base:
                conditional_tree = FPTree()
                item_counts = defaultdict(int)
                for path in conditional_base:
                    for item in path:
                        item_counts[item] += 1
                filtered_items = {item for item, count in item_counts.items() if count >= min_support_count}
                for path in conditional_base:
                    filtered_path = [item for item in path if item in filtered_items]
                    if filtered_path:
                        filtered_path.sort(key=lambda x: item_counts[x], reverse=True)
                        current_node = conditional_tree.root
                        for item in filtered_path:
                            if item in current_node.children:
                                current_node.children[item].count += 1
                            else:
                                new_node = FPNode(item, 1, current_node)
                                current_node.children[item] = new_node
                                conditional_tree.header_table[item].append(new_node)
                            current_node = current_node.children[item]
                mine_patterns(conditional_tree.header_table, min_support_count, new_prefix)
    
    mine_patterns(fp_tree.header_table, min_support_count, set())
    return frequent_itemsets

# Custom Distance Calculations
def custom_euclidean(p1, p2):
    return np.sqrt(sum((a - b) ** 2 for a, b in zip(p1, p2)))

def custom_manhattan(p1, p2):
    return sum(abs(a - b) for a, b in zip(p1, p2))

def custom_chebyshev(p1, p2):
    return max(abs(a - b) for a, b in zip(p1, p2))

# Example Usage
if __name__ == "__main__":
    # Sample transaction data
    transactions = [
        ['milk', 'bread', 'butter'],
        ['milk', 'bread'],
        ['milk', 'butter'],
        ['bread', 'butter'],
        ['milk', 'bread', 'butter', 'cheese']
    ]
    min_support = 0.4

    # Apriori and FP-Growth results
    print("Apriori Frequent Itemsets:")
    for itemset, count in apriori(transactions, min_support).items():
        print(f"{set(itemset)}: {count}")
    
    print("\nFP-Growth Frequent Itemsets:")
    for itemset, count in fp_growth(transactions, min_support).items():
        print(f"{set(itemset)}: {count}")

    # Distance calculations for points [2,3] and [5,7]
    p1, p2 = [2, 3], [5, 7]
    print("\nDistance Calculations:")
    print("Custom:")
    print(f"Euclidean: {custom_euclidean(p1, p2):.2f}")
    print(f"Manhattan: {custom_manhattan(p1, p2):.2f}")
    print(f"Chebyshev: {custom_chebyshev(p1, p2):.2f} (max(|2-5|, |3-7|) = max(3,4) = 4)")
    print("Scipy:")
    print(f"Euclidean: {distance.euclidean(p1, p2):.2f}")from scipy.spatial import distance
import numpy as np

def calculate_distances(point1, point2):
    # Convert points to numpy arrays
    p1 = np.array(point1)
    p2 = np.array(point2)
    
    # Calculate distances
    euclidean_dist = distance.euclidean(p1, p2)
    manhattan_dist = distance.cityblock(p1, p2)
    chebyshev_dist = distance.chebyshev(p1, p2)
    
    # For Chebyshev explanation
    abs_diffs = np.abs(p1 - p2)
    chebyshev_manual = np.max(abs_diffs)
    
    return {
        'euclidean': euclidean_dist,
        'manhattan': manhattan_dist,
        'chebyshev': chebyshev_dist,
        'chebyshev_components': list(abs_diffs),
        'chebyshev_max': chebyshev_manual
    }

# Example usage
if __name__ == "__main__":
    point1 = [2, 3]
    point2 = [5, 7]
    distances = calculate_distances(point1, point2)
    
    print(f"Points: {point1} and {point2}")
    print(f"Euclidean Distance: {distances['euclidean']:.2f}")
    print(f"Manhattan Distance: {distances['manhattan']:.2f}")
    print(f"Chebyshev Distance: {distances['chebyshev']:.2f}")
    print(f"Chebyshev Components: |{point1[0]}-{point2[0]}|={distances['chebyshev_components'][0]}, "
          f"|{point1[1]}-{point2[1]}|={distances['chebyshev_components'][1]}")
    print(f"Chebyshev Max: max{distances['chebyshev_components']} = {distances['chebyshev_max']}")
    print(f"Manhattan: {distance.cityblock(p1, p2):.2f}")
    print(f"Chebyshev: {distance.chebyshev(p1, p2):.2f}")

Apriori Frequent Itemsets:
{'milk'}: 4
{'bread'}: 4
{'butter'}: 4
{'bread', 'milk'}: 3
{'bread', 'butter'}: 3
{'milk', 'butter'}: 3
{'bread', 'milk', 'butter'}: 2

FP-Growth Frequent Itemsets:
{'milk'}: 4
{'bread'}: 4
{'bread', 'milk'}: 3
{'butter'}: 4
{'bread', 'butter'}: 3
{'milk', 'butter'}: 3
{'bread', 'milk', 'butter'}: 2

Distance Calculations:
Custom:
Euclidean: 5.00
Manhattan: 7.00
Chebyshev: 4.00 (max(|2-5|, |3-7|) = max(3,4) = 4)
Scipy:
Euclidean: 5.00
Manhattan: 7.00
Chebyshev: 4.00


In [6]:
from itertools import combinations
from collections import defaultdict

def apriori(transactions, min_support=0.2):
    # Count individual items
    item_count = defaultdict(int)
    for transaction in transactions:
        for item in transaction:
            item_count[item] += 1
    n_transactions = len(transactions)
    frequent_items = {frozenset([item]): count/n_transactions 
                     for item, count in item_count.items() 
                     if count/n_transactions >= min_support}
    k = 2
    frequent_itemsets = frequent_items.copy()
    while True:
        candidates = defaultdict(int)
        for itemset1, itemset2 in combinations(frequent_items.keys(), 2):
            union = itemset1 | itemset2
            if len(union) == k:
                candidates[union] += sum(1 for t in transactions if union.issubset(t))
        new_frequent = {itemset: count/n_transactions 
                       for itemset, count in candidates.items() 
                       if count/n_transactions >= min_support}
        
        if not new_frequent:
            break
            
        frequent_itemsets.update(new_frequent)
        frequent_items = new_frequent
        k += 1
    rules = []
    for itemset in frequent_itemsets:
        if len(itemset) > 1:
            for i in range(1, len(itemset)):
                for antecedent in combinations(itemset, i):
                    antecedent = frozenset(antecedent)
                    consequent = itemset - antecedent
                    if antecedent in frequent_itemsets:
                        confidence = frequent_itemsets[itemset] / frequent_itemsets[antecedent]
                        if confidence >= 0.7:
                            rules.append({
                                'antecedent': set(antecedent),
                                'consequent': set(consequent),
                                'support': frequent_itemsets[itemset],
                                'confidence': confidence
                            })
    
    return frequent_itemsets, rules
if __name__ == "__main__":
    transactions = [
        ['bread', 'milk', 'butter'],
        ['bread', 'milk'],
        ['milk', 'butter'],
        ['bread', 'butter'],
        ['bread', 'milk', 'butter', 'cheese']
    ]
    itemsets, rules = apriori(transactions, min_support=0.2)
    print("Frequent Itemsets:")
    for itemset, support in itemsets.items():
        print(f"{set(itemset)}: {support:.2f}")
    print("\nAssociation Rules:")
    for rule in rules:
        print(f"{rule['antecedent']} => {rule['consequent']}: "
              f"support={rule['support']:.2f}, confidence={rule['confidence']:.2f}")

Frequent Itemsets:
{'bread'}: 0.80
{'milk'}: 0.80
{'butter'}: 0.80
{'cheese'}: 0.20
{'bread', 'milk'}: 0.60
{'bread', 'butter'}: 0.60
{'bread', 'cheese'}: 0.20
{'milk', 'butter'}: 0.60
{'cheese', 'milk'}: 0.20
{'cheese', 'butter'}: 0.20
{'bread', 'milk', 'butter'}: 1.20
{'bread', 'milk', 'cheese'}: 0.60
{'bread', 'cheese', 'butter'}: 0.60
{'cheese', 'milk', 'butter'}: 0.60
{'butter', 'bread', 'milk', 'cheese'}: 1.20

Association Rules:
{'bread'} => {'milk'}: support=0.60, confidence=0.75
{'milk'} => {'bread'}: support=0.60, confidence=0.75
{'bread'} => {'butter'}: support=0.60, confidence=0.75
{'butter'} => {'bread'}: support=0.60, confidence=0.75
{'cheese'} => {'bread'}: support=0.20, confidence=1.00
{'milk'} => {'butter'}: support=0.60, confidence=0.75
{'butter'} => {'milk'}: support=0.60, confidence=0.75
{'cheese'} => {'milk'}: support=0.20, confidence=1.00
{'cheese'} => {'butter'}: support=0.20, confidence=1.00
{'bread'} => {'milk', 'butter'}: support=1.20, confidence=1.50
{'milk'}

In [7]:
import pandas as pd
from mlxtend.frequent_patterns import fpgrowth, association_rules

# Load and preprocess CSV file
def process_fpgrowth(csv_file, min_support=0.2, min_confidence=0.7):
    # Read CSV
    df = pd.read_csv(csv_file)
    
    # Convert to binary matrix if needed
    # Assuming CSV contains transaction data
    if df.dtypes[0] == 'object':  # If data is in list format
        # One-hot encode the transactions
        transactions = df.iloc[:, 0].str.split(',', expand=True)
        df_encoded = pd.get_dummies(transactions.stack()).groupby(level=0).sum()
    else:
        df_encoded = df  # Assume already encoded
    
    # Apply FP-Growth
    fp = fpgrowth(df_encoded, min_support=min_support, use_colnames=True)
    
    # Generate association rules
    rules = association_rules(fp, metric="confidence", min_threshold=min_confidence)
    
    return fp, rules

# Example usage
if __name__ == "__main__":
    # Example CSV content:
    # transactions
    # bread,milk,butter
    # bread,milk
    # milk,butter
    # bread,butter
    # bread,milk,butter,cheese
    try:
        frequent_itemsets, rules = process_fpgrowth('transactions.csv')
        print("Frequent Itemsets:")
        print(frequent_itemsets)
        print("\nAssociation Rules:")
        print(rules[['antecedents', 'consequents', 'support', 'confidence']])
    except FileNotFoundError:
        print("Please provide a valid transactions.csv file")

Please provide a valid transactions.csv file


In [8]:
from scipy.spatial import distance
import numpy as np
def calculate_distances(point1, point2):
    p1 = np.array(point1)
    p2 = np.array(point2)
    euclidean_dist = distance.euclidean(p1, p2)
    manhattan_dist = distance.cityblock(p1, p2)
    chebyshev_dist = distance.chebyshev(p1, p2)
    abs_diffs = np.abs(p1 - p2)
    chebyshev_manual = np.max(abs_diffs)
    
    return {
        'euclidean': euclidean_dist,
        'manhattan': manhattan_dist,
        'chebyshev': chebyshev_dist,
        'chebyshev_components': list(abs_diffs),
        'chebyshev_max': chebyshev_manual
    }
if __name__ == "__main__":
    point1 = [2, 3]
    point2 = [5, 7]
    distances = calculate_distances(point1, point2)
    
    print(f"Points: {point1} and {point2}")
    print(f"Euclidean Distance: {distances['euclidean']:.2f}")
    print(f"Manhattan Distance: {distances['manhattan']:.2f}")
    print(f"Chebyshev Distance: {distances['chebyshev']:.2f}")
    print(f"Chebyshev Components: |{point1[0]}-{point2[0]}|={distances['chebyshev_components'][0]}, "
          f"|{point1[1]}-{point2[1]}|={distances['chebyshev_components'][1]}")
    print(f"Chebyshev Max: max{distances['chebyshev_components']} = {distances['chebyshev_max']}")

Points: [2, 3] and [5, 7]
Euclidean Distance: 5.00
Manhattan Distance: 7.00
Chebyshev Distance: 4.00
Chebyshev Components: |2-5|=3, |3-7|=4
Chebyshev Max: max[3, 4] = 4
